# Проверка: объёмная сила на стенку vs тензор Максвелла (MenDrive_qnm.ipynb, `driving_mode_1 = 'je_y, k_y=0, TE'`)

Ноутбук независим от Sage-символики: формулы ячеек 101–131, 232, 291–347, 381–396, 476 ноутбука `MenDrive_qnm.ipynb`
переписаны на `numpy` (`longdouble`, ≈33 знака на x86-64). Каждая пара — Markdown с формулами → Code с проверкой **OK/FAIL** и невязкой.

Запуск: `python3 -m pip install numpy jupyter` и `jupyter nbconvert --to notebook --execute check_force_tensor_vs_volume.ipynb`,
или в ядре SageMath (первая ячейка отключает препарсер Sage).

Обозначения: СГС, $e^{-i\omega t}$, $e^{ik_z z}$, $k_z=\pi/L_z$, $k_y=0$; левая стенка $x<-a$, вакуум $|x|\le a$, правая стенка $x>a$;
$\varepsilon'=\varepsilon+4\pi i\sigma_e/\omega$, $\mu'=\mu+4\pi i\sigma_m/\omega$; усреднение по периоду $\langle AB\rangle=\tfrac12\mathrm{Re}(A B^*)$.

In [ ]:
try:
    preparser(False)      # ядро SageMath: дальше обычный Python
except NameError:
    pass

import numpy as np
LD = np.longdouble; CD = np.clongdouble
pi = LD('3.14159265358979323846264338327950288'); I = CD(1j)
# ---- параметры MenDrive_qnm.ipynb (ячейки 6, 10, 22, 23; вывод ячейки 420) ----
P = dict(a=LD('0.5'), c=LD(29979245800), Lz=LD(100),
         eps_l=LD(1), mu_l=LD(1), se_l=LD('5.625e17'), sm_l=LD(0),
         eps_r=LD(16), mu_r=LD(100), se_r=LD('9.9e12'), sm_r=LD('6.48e7'),
         h_l=LD('0.0001875'), h_r=LD('0.00625'))
def media(w, p=P):
    """eps' = eps + 4 pi i sigma_e/w,  mu' = mu + 4 pi i sigma_m/w;  K (Im K > 0), K_v = sqrt(w^2/c^2 - k_z^2)."""
    w = LD(w); c = p['c']; kz = pi/p['Lz']
    el = p['eps_l'] + 4*pi*I*p['se_l']/w; ml = p['mu_l'] + 4*pi*I*p['sm_l']/w
    er = p['eps_r'] + 4*pi*I*p['se_r']/w; mr = p['mu_r'] + 4*pi*I*p['sm_r']/w
    Kv = np.sqrt(CD(w**2/c**2 - kz**2))
    def KK(e, m):
        K = np.sqrt(CD(-(kz**2 - e*m*w**2/c**2)))   # K2_x_TE при k_y=0, mu_xx=mu_zz
        return K if K.imag >= 0 else -K
    return dict(c=c, kz=kz, el=el, ml=ml, er=er, mr=mr, Kv=Kv, Kl=KK(el, ml), Kr=KK(er, mr), w=w)

def fields_nb(m, X, je0=0, p=P):
    """Поля РОВНО как в ноутбуке (ячейки 101-131), X = [B_H_zl, A1_H_z, A2_H_z, B_H_zr], sign_K = 1."""
    c, kz, w, Kv, Kl, Kr = m['c'], m['kz'], m['w'], m['Kv'], m['Kl'], m['Kr']
    el, ml, er, mr = m['el'], m['ml'], m['er'], m['mr']; a = p['a']
    Bl, A1, A2, Br = X
    dv = c**2*kz**2 - w**2; dl = c**2*kz**2 - el*ml*w**2; dr = c**2*kz**2 - er*mr*w**2
    def vac(x):
        e1 = np.exp(-I*Kv*x); e2 = np.exp(I*Kv*x)
        Hz = A1*e1 + A2*e2; Hx = c**2*kz*Kv*(A2*e2 - A1*e1)/dv; Ey = Kv*c*w*(A1*e1 - A2*e2)/dv
        return dict(Ey=Ey, Hx=Hx, Hz=Hz, Dy=Ey, Bx=Hx, Bz=Hz)
    def left(x):
        ph = np.exp(-I*Kl*(x + a))
        Hz = Bl*ph; Hx = -(Bl*Kl*c**2*kz + 4*pi*I*c*je0*kz)*ph/dl; Ey = (Bl*Kl*c*ml*w + 4*pi*I*je0*ml*w)*ph/dl
        return dict(Ey=Ey, Hx=Hx, Hz=Hz, Dy=el*Ey, Bx=ml*Hx, Bz=ml*Hz)
    def right(x):
        ph = np.exp(I*Kr*(x - a))
        Hz = Br*ph; Hx = Br*Kr*c**2*kz*ph/dr; Ey = -Br*Kr*c*mr*w*ph/dr
        return dict(Ey=Ey, Hx=Hx, Hz=Hz, Dy=er*Ey, Bx=mr*Hx, Bz=mr*Hz)
    return vac, left, right

def M4(m, p=P):
    """M4_K (ячейка 232) и B4_K/je0 (свободный член eq_Ey_l)."""
    c, kz, w, Kv, Kl, Kr = m['c'], m['kz'], m['w'], m['Kv'], m['Kl'], m['Kr']
    el, ml, er, mr = m['el'], m['ml'], m['er'], m['mr']; a = p['a']
    dv = c**2*kz**2 - w**2; dl = c**2*kz**2 - el*ml*w**2; dr = c**2*kz**2 - er*mr*w**2
    M = np.zeros((4, 4), dtype=CD)
    M[0, 0] = 1; M[0, 1] = -np.exp(I*Kv*a); M[0, 2] = -np.exp(-I*Kv*a)
    M[1, 3] = 1; M[1, 1] = -np.exp(-I*Kv*a); M[1, 2] = -np.exp(I*Kv*a)
    M[2, 0] = Kl*c*ml*w/dl; M[2, 1] = -Kv*c*w*np.exp(I*Kv*a)/dv; M[2, 2] = Kv*c*w*np.exp(-I*Kv*a)/dv
    M[3, 3] = -Kr*c*mr*w/dr; M[3, 1] = -Kv*c*w*np.exp(-I*Kv*a)/dv; M[3, 2] = Kv*c*w*np.exp(I*Kv*a)/dv
    Bv = np.array([0, 0, -4*pi*I*ml*w/dl, 0], dtype=CD)
    return M, Bv

def svd_X4(M):
    """mendrivepylib.common.svd_X4: минимальный сингулярный вектор ОДНОРОДНОЙ системы (B_ext игнорируется)."""
    Mn = M.astype(np.complex128)
    U, s, Vh = np.linalg.svd(Mn); X = Vh[-1]
    Xs = np.array([X[0], X[2], X[1], X[3]])
    cands = [X, 1j*X, -1j*X, np.conj(X), Xs, 1j*Xs, -1j*Xs, np.conj(Xs)]
    best = min(cands, key=lambda v: np.linalg.norm(Mn @ (v/np.linalg.norm(v))))
    best = best/np.linalg.norm(best)
    return best.astype(CD), float(np.linalg.norm(Mn @ best)), s

def gauss_solve(A, b):
    A = A.copy(); b = b.copy(); n = len(b)
    for k in range(n):
        piv = k + int(np.argmax([abs(A[i, k]) for i in range(k, n)]))
        A[[k, piv]] = A[[piv, k]]; b[[k, piv]] = b[[piv, k]]
        for i in range(k + 1, n):
            f = A[i, k]/A[k, k]; A[i, k:] -= f*A[k, k:]; b[i] -= f*b[k]
    x = np.zeros(n, dtype=CD)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - np.sum(A[i, i + 1:]*x[i + 1:]))/A[i, i]
    return x

def Txx(F):
    """<T_xx> Минковского (Тамм 105.1, compute_maxwell_stress_tensor_symbolic), усреднение по периоду."""
    return np.real(2*F['Hx']*np.conj(F['Bx']) - F['Ey']*np.conj(F['Dy'])
                   - F['Hx']*np.conj(F['Bx']) - F['Hz']*np.conj(F['Bz']))/(16*pi)

def nb_terms(w, je0=LD(1e6), p=P, conj_bug=True):
    """12+12 слагаемых force_dyn_right/left из calc_thrust (ячейка 396) на полях svd_X4 и W_total (слои h)."""
    m = media(w, p); M, Bv = M4(m, p); X, res, _ = svd_X4(M)
    vac, left, right = fields_nb(m, X, je0, p)
    c, ww = m['c'], m['w']; a = p['a']
    el, ml, er, mr = m['el'], m['ml'], m['er'], m['mr']; Kl, Kr = m['Kl'], m['Kr']; kz = m['kz']
    R = right(a); L = left(-a); Va = vac(a); Vm = vac(-a)
    # E_y_l(x), H_x_l(x) ячейки 278 (матрица M_left): je0-член входит И в столбец 'Ez', И в столбец 'Hz' -> удвоен (8 pi)
    L2 = fields_nb(m, X, 2*je0, p)[1](-a)
    iR = 1/(2*Kr.imag); iL = 1/(2*Kl.imag)
    t = {}
    t['F_jmD_conj_right']      = np.real(p['sm_r']*R['Hz']*np.conj(er*R['Ey']))/(2*c)*iR
    t['p_Bt_jeB_conj_right']   = np.real(p['se_r']*R['Ey']*np.conj(mr*R['Hz']))/(2*c)*iR
    t['p_I_conv_Ht_conj_right']= np.real((1 - mr)*Va['Hz']*np.conj(Va['Hz']))/(16*pi)
    t['t_I_conv_Hn_conj_right']= np.real((1 - mr)*R['Hx']*np.conj(R['Hx']))/(16*pi)
    if conj_bug:   # как получилось в ячейках 293/291 после .simplify(): conj(mu'_r) -> mu'_r
        conjEy = -np.conj(X[3])*np.conj(Kr)*c*mr*ww/(c**2*kz**2 - np.conj(er)*mr*ww**2)
        t['F_I_rotB_conj_r'] = np.real(-(mr - 1)/(4*pi)*R['Hz']*I*ww/c*mr*np.conj(er)*conjEy)/2*iR
        t['F_P_rotE_conj_r'] = np.real(-(er - 1)/(4*pi)*R['Ey']*I*ww/c*mr*np.conj(R['Hz']))/2*iR
    else:
        t['F_I_rotB_conj_r'] = np.real(-(mr - 1)/(4*pi)*R['Hz']*I*ww/c*np.conj(mr*er*R['Ey']))/2*iR
        t['F_P_rotE_conj_r'] = np.real(-(er - 1)/(4*pi)*R['Ey']*I*ww/c*np.conj(mr*R['Hz']))/2*iR
    t['F_dyn_I_D_r']           = np.real(I*ww/c*(mr - 1)/(4*pi)*R['Hz']*np.conj(er*R['Ey']))/2*iR
    t['t_Dn_conj_r']           = LD(0)
    t['p_Et_conj_r']           = np.real((1 - er)*Va['Ey']*np.conj(Va['Ey']))/(16*pi)
    t['F_dyn_P_B_r']           = np.real(-I*ww/c*(er - 1)/(4*pi)*R['Ey']*np.conj(mr*R['Hz']))/2*iR
    t['f_jump_maxwell_right']  = Txx(R) - Txx(Va)
    t['F_jm_ext_D_conj_right'] = LD(0)
    u = {}
    u['p_Bt_jeB_conj_left']    = np.real(p['se_l']*L2['Ey']*np.conj(ml*L2['Hz']))/(2*c)*iL
    u['F_P_rotE_conj_l']       = np.real(-(el - 1)/(4*pi)*L2['Ey']*I*ww/c*np.conj(ml*L2['Hz']))/2*iL
    u['F_dyn_P_B_l']           = np.real(-I*ww/c*(el - 1)/(4*pi)*L2['Ey']*np.conj(ml*L2['Hz']))/2*iL
    u_cell110 = np.real(p['se_l']*L['Ey']*np.conj(ml*L['Hz']))/(2*c)*iL   # с полями ячейки 110 (4 pi)
    u['p_Et_conj_l']           = np.real((el - 1)*Vm['Ey']*np.conj(Vm['Ey']))/(16*pi)
    u['f_jump_maxwell_left']   = Txx(Vm) - Txx(L)
    u['p_Bt_je_ext_B_conj_left'] = np.real(je0*np.conj(ml*L['Hz']))/(2*c)*iL
    gR = (1 - np.exp(-2*Kr.imag*p['h_r']))*iR; gL = (1 - np.exp(-2*Kl.imag*p['h_l']))*iL
    Wr = (p['se_r']*abs(R['Ey'])**2 + p['sm_r']*(abs(R['Hx'])**2 + abs(R['Hz'])**2))/2*gR
    Wl = (p['se_l']*abs(L['Ey'])**2)/2*gL
    W = Wr + Wl
    nk = lambda f: f*LD(1e5)/W          # Н/кВт = (dyn/cm^2)/(erg/s/cm^2) * 1e5
    return dict(m=m, X=X, res=res, W=W, t=t, u=u, nk=nk, Tv_a=Txx(Va), Tv_ma=Txx(Vm), Tr_a=Txx(R), Tl_ma=Txx(L), u_cell110=u_cell110,
                R=R, Va=Va, iR=iR)

GLx, GLw = np.polynomial.legendre.leggauss(64)
def gl_int(f, x0, x1, npan=200):
    edges = np.linspace(float(x0), float(x1), npan + 1); s = CD(0)
    for i in range(npan):
        A_, B_ = LD(edges[i]), LD(edges[i + 1]); h = (B_ - A_)/2; mid = (A_ + B_)/2
        s += h*np.sum(GLw.astype(LD)*f(mid + h*GLx.astype(LD)))
    return s

def solve_driven(w, je0=LD(1e6), p=P):
    """Точное решение TE (k_y=0, exp(i k_z z - i w t)) с je_y = je0*exp(-i K_l (x+a)) в левой стенке.
    u = E_y:  u'' + K^2 u = -(4 pi i w mu'_zz/c^2) j;   H_z = c u'/(i w mu'_zz);   H_x = -c k_z u/(w mu'_xx).
    Источник резонансный (K_s = K_l)  ->  секулярный член  u_p = alpha*(x+a)*exp(-i K_l (x+a))."""
    m = media(w, p); c, kz, ww = m['c'], m['kz'], m['w']; a = p['a']
    Kv, Kl, Kr = m['Kv'], m['Kl'], m['Kr']; ml, mr = m['ml'], m['mr']
    g = -(4*pi*I*ww*ml/c**2)*je0; alpha = g/(-2*I*Kl)
    up = lambda s: alpha*s*np.exp(-I*Kl*s); dup = lambda s: alpha*np.exp(-I*Kl*s)*(1 - I*Kl*s)
    A = np.zeros((4, 4), dtype=CD); b = np.zeros(4, dtype=CD)
    em, ep = np.exp(I*Kv*a), np.exp(-I*Kv*a)
    A[0] = [1, -em, -ep, 0];                         b[0] = -up(LD(0))
    A[1] = [-I*Kl/ml, I*Kv*em, -I*Kv*ep, 0];         b[1] = -dup(LD(0))/ml
    A[2] = [0, ep, em, -1]
    A[3] = [0, -I*Kv*ep, I*Kv*em, -I*Kr/mr]
    X = gauss_solve(A, b); Cl, Pp, Qq, Cr = X
    def U(x, reg):
        if reg == 'l':
            s = x + a; u = Cl*np.exp(-I*Kl*s) + up(s); du = -I*Kl*Cl*np.exp(-I*Kl*s) + dup(s)
            mz = ml; eps = p['eps_l']; se = p['se_l']; sm = p['sm_l']; mu = p['mu_l']; jx = je0*np.exp(-I*Kl*s)
        elif reg == 'v':
            u = Pp*np.exp(-I*Kv*x) + Qq*np.exp(I*Kv*x); du = -I*Kv*Pp*np.exp(-I*Kv*x) + I*Kv*Qq*np.exp(I*Kv*x)
            mz = CD(1); eps = LD(1); se = sm = LD(0); mu = LD(1); jx = 0*x
        else:
            u = Cr*np.exp(I*Kr*(x - a)); du = I*Kr*u; mz = mr; eps = p['eps_r']; se = p['se_r']; sm = p['sm_r']; mu = p['mu_r']; jx = 0*x
        Hz = c*du/(I*ww*mz); Hx = -c*kz*u/(ww*mz)
        return dict(Ey=u, Hz=Hz, Hx=Hx, eps=eps, mu=mu, se=se, sm=sm, jext=jx, Dy=eps*u, Bz=mu*Hz, Bx=mu*Hx,
                    dHx=-c*kz*du/(ww*mz))
    return dict(m=m, X=X, U=U, A=A, b=b)

def fM(F, c):
    """Минковский: <f_x> = (1/2c) Re[(sigma_e E_y + j_ext) B_z^*] + (1/2c) Re[sigma_m H_z D_y^*],  B = mu H, D = eps E (вещественные)."""
    je = F['se']*F['Ey'] + F['jext']; jm = F['sm']*F['Hz']
    return (np.real(je*np.conj(F['Bz'])) + np.real(jm*np.conj(F['Dy'])))/(2*c)
def TEL(F):
    """<T_xx> Эйнштейна-Лауба: (1/16 pi) Re[2 B_x H_x^* - |E|^2 - |H|^2]."""
    return np.real(2*F['Bx']*np.conj(F['Hx']) - abs(F['Ey'])**2 - abs(F['Hx'])**2 - abs(F['Hz'])**2)/(16*pi)
def fEL(F, c, w, kz):
    """Эйнштейн-Лауб: <(M.grad)H>_x + (1/2c)Re[(j_e + dP/dt) H_z^*] + (1/2c)Re[(j_m + dM_z/dt) E_y^*]."""
    Pp = (F['eps'] - 1)/(4*pi)*F['Ey']; Mx = (F['mu'] - 1)/(4*pi)*F['Hx']; Mz = (F['mu'] - 1)/(4*pi)*F['Hz']
    Je = F['se']*F['Ey'] + F['jext'] - I*w*Pp; Jm = F['sm']*F['Hz'] - I*w*Mz
    MgradH = np.real(Mx*np.conj(F['dHx']) + Mz*np.conj(I*kz*F['Hx']))/2
    return MgradH + np.real(Je*np.conj(F['Hz']))/(2*c) + np.real(Jm*np.conj(F['Ey']))/(2*c)

def balance(w, je0=LD(1e6), p=P):
    S = solve_driven(w, je0, p); m = S['m']; U = S['U']; a = p['a']; c = m['c']; ww = m['w']; kz = m['kz']
    Ll = 40/float(m['Kl'].imag); Lr = 40/float(m['Kr'].imag)
    r = {}
    r['Fr_vol_M'] = np.real(gl_int(lambda x: fM(U(x, 'r'), c), a, a + Lr))
    r['Fl_vol_M'] = np.real(gl_int(lambda x: fM(U(x, 'l'), c), -a - Ll, -a))
    r['Fr_vol_EL'] = np.real(gl_int(lambda x: fEL(U(x, 'r'), c, ww, kz), a, a + Lr))
    r['Tl'] = Txx(U(-a, 'l')); r['Tvl'] = Txx(U(-a, 'v')); r['Tvr'] = Txx(U(a, 'v')); r['Tr'] = Txx(U(a, 'r'))
    r['Tr_EL'] = TEL(U(a, 'r'))
    r['Fr_surf'] = r['Tr'] - r['Tvr']; r['Fl_surf'] = r['Tvl'] - r['Tl']
    r['Fr_tot'] = r['Fr_vol_M'] + r['Fr_surf']; r['Fl_tot'] = r['Fl_vol_M'] + r['Fl_surf']
    r['thrust'] = r['Fr_tot'] + r['Fl_tot']
    r['Psrc'] = -np.real(gl_int(lambda x: np.conj(U(x, 'l')['jext'])*U(x, 'l')['Ey'], -a - Ll, -a))/2
    pd = lambda F: (F['se']*abs(F['Ey'])**2 + F['sm']*(abs(F['Hx'])**2 + abs(F['Hz'])**2))/2
    r['Pdiss'] = np.real(gl_int(lambda x: pd(U(x, 'l')), -a - Ll, -a)) + np.real(gl_int(lambda x: pd(U(x, 'r')), a, a + Lr))
    r['S'] = S
    return r

def ok(name, val, ref, tol):
    err = abs(float(val) - float(ref))/max(abs(float(ref)), 1e-300)
    print(f"{'OK  ' if err <= tol else 'FAIL'} {name}: {float(val):.12g}  (эталон {float(ref):.12g}, невязка {err:.2e}, допуск {tol:.0e})")
    return err <= tol

NB_REF = {1000000.0: (3466.3287444014, -0.00712182972917462, -0.00343098992775451, 0.0009803919304189684), 2000000.0: (857.470293777726, 0.00294895200222664, -0.0133503044383414, 0.0011199492629390823), 3000000.0: (373.962657080211, 0.0041582449405522, -0.0143772992695921, 0.0011683779770726137), 4000000.0: (204.350667643417, 0.00445748455335753, -0.0144472992865367, 0.001209369515356029), 5000000.0: (125.512113499692, 0.00451014809951942, -0.0142345791131933, 0.001252369835513554), 6000000.0: (82.4240660027207, 0.00446531654697287, -0.0139006015723876, 0.0012986108031769555), 7000000.0: (56.2515109971598, 0.00437462820139094, -0.013507903962563, 0.0013477303314664423), 8000000.0: (39.1341056313676, 0.00426109248829533, -0.0130881591138931, 0.0013990535291182984), 9000000.0: (27.317459416666, 0.00413666744249033, -0.012659752437143, 0.0014519253535879067), 10000000.0: (18.8216555968199, 0.00400814712635522, -0.0122339747794219, 0.001505797513514477)}   # вывод ячейки 479: thrust_N_per_kW_right, N_per_kW_maxwell_right_wall, thrust_N_per_kW_left, residual
print('ядро загружено; longdouble eps =', np.finfo(LD).eps)

## 1. Что суммирует `calc_thrust` (ячейка 396)

$$F^{nb}_{right}=\underbrace{F_{jmD}}_{\frac{1}{2c}\mathrm{Re}\!\int\sigma_m H_z(\varepsilon' E_y)^*}+\underbrace{p_{Bt,jeB}}_{\frac{1}{2c}\mathrm{Re}\!\int\sigma_e E_y(\mu' H_z)^*}
+\underbrace{p_{I,conv,Ht}+t_{I,conv,Hn}+p_{Et}+t_{Dn}}_{\text{«Смайт», ячейки 381–390}}
+\underbrace{F_{I\times\mathrm{rot}B}+F_{dyn,I,D}+F_{P\times\mathrm{rot}E}+F_{dyn,P,B}}_{\text{«Абрагам», ячейки 291–319}}
+\underbrace{[T_{xx}]_{a}}_{f\_jump\_maxwell\_right}+F_{jm,ext,D},$$

все интегралы $\int_a^{\infty}$ (берутся как $\int_a^\infty e^{-2\,\mathrm{Im}K_r(x-a)}dx=\frac{1}{2\mathrm{Im}K_r}$), поля — минимальный
сингулярный вектор **однородной** системы `svd_X4(M4)`, нормировка «на кВт» — на $W=\int_{-a-h_l}^{-a}+\int_a^{a+h_r}\tfrac12(\sigma_e|E|^2+\sigma_m|H|^2)dx$
(только слои толщины $h_l,h_r$), Н/кВт $=10^5\,F[\text{дин/см}^2]/W[\text{эрг/(с·см}^2)]$.

Замечание: в формулах сил левой стенки (ячейки 278 → 291–347) $E_y$ берётся из матрицы `M_left`, где член с `je0_ext` стоит и в столбце `'Ez'`, и в столбце `'Hz'`, поэтому он **удвоен** ($8\pi$ вместо $4\pi$ ячейки 110; правильно — $2\pi$, см. §5). Это учтено, чтобы воспроизвести вывод.

Проверка: воспроизводим вывод ячейки 479 (тяга правой стенки, тензор на правой стенке, левая стенка, невязка SVD).

In [ ]:
allok = True
for w, (nb_r, nb_mx, nb_l, nb_res) in sorted(NB_REF.items()):
    o = nb_terms(w)
    Fr = sum(o['t'].values()); Fl = sum(o['u'].values())
    print(f"omega = {w:.1e}")
    allok &= ok('  thrust_N_per_kW_right', o['nk'](Fr), nb_r, 1e-9)
    allok &= ok('  N_per_kW_maxwell_right_wall = -T_v(a)', o['nk'](-o['Tv_a']), nb_mx, 1e-9)
    allok &= ok('  thrust_N_per_kW_left', o['nk'](Fl), nb_l, 1e-9)
    allok &= ok('  residual svd_X4', o['res'], nb_res, 1e-9)
print('ИТОГ:', 'OK — ноутбук воспроизведён' if allok else 'FAIL')

## 2. Откуда 3466 Н/кВт: потерянное сопряжение $\mu'_r$ в $F_{I\times\mathrm{rot}B}$

В ячейке 293 $f_{I\times\mathrm{rot}B}=-\tfrac12\,\frac{\mu'_{zz}-1}{4\pi}H_z\,\frac{i\omega}{c}\,\big(\mu'_{zz}\varepsilon'_{yy}E_y\big)^*$, затем `.simplify()`.
Символы `mu_r_xx_c, mu_r_zz_c` в `mendrivepylib/variables/common.py` **не объявлены** `complex` (объявлены только `mu_l_*_c` и `epsilon_*_c`),
Maxima считает их вещественными и выбрасывает `conjugate` (в выводе ячеек 291, 293, 303 справа стоит `mu_r_zz_c^2`, `mu_r_xx_c` без `conjugate`,
слева — `conjugate(mu_l_zz_c)`). После подстановки $\mu'_r=100+4\pi i\sigma_m/\omega$ получаем вместо $|\mu'|^2$ величину $\mu'^2$.

При $\mathrm{rot}\,H_y\simeq -\partial_xH_z=-iK_rH_z$, $\mu''=4\pi\sigma_m/\omega$, $K_r=K'+iK''$:

$$F^{ok}_{I\times\mathrm{rot}B}=-\frac{|H_z(a)|^2}{16\pi}\Big[\mu(\mu-1)+\mu''^2-\mu''\tfrac{K'}{K''}\Big],\qquad
F^{bug}_{I\times\mathrm{rot}B}=-\frac{|H_z(a)|^2}{16\pi}\Big[\mu(\mu-1)-\mu''^2-\mu''(2\mu-1)\tfrac{K'}{K''}\Big].$$

При $\omega=10^6$: $\mu''=814$, $\mu''^2=6.6\cdot10^5\gg\mu(\mu-1)=9900$ — отсюда знак «+», величина $\sim\mu''^2$ раз больше давления $|H|^2/16\pi$
и закон $\propto 1/\omega^2$.

In [ ]:
allok = True
for w in [1e6, 3e6, 1e7]:
    o = nb_terms(w); m = o['m']; Kr = m['Kr']; H2 = abs(o['R']['Hz'])**2
    mu2 = 4*pi*P['sm_r']/LD(w); mu = P['mu_r']
    th_bug = -(H2/(16*pi))*(mu*(mu - 1) - mu2**2 - mu2*(2*mu - 1)*Kr.real/Kr.imag)
    th_ok  = -(H2/(16*pi))*(mu*(mu - 1) + mu2**2 - mu2*Kr.real/Kr.imag)
    o_ok = nb_terms(w, conj_bug=False)
    print(f"omega={w:.0e}: F_I_rotB(bug) = {float(o['nk'](o['t']['F_I_rotB_conj_r'])):.6g} Н/кВт,  F_I_rotB(ok) = {float(o_ok['nk'](o_ok['t']['F_I_rotB_conj_r'])):.6g} Н/кВт")
    allok &= ok('  bug-член vs асимптотика', o['t']['F_I_rotB_conj_r'], th_bug, 1e-4)
    allok &= ok('  ok-член vs асимптотика', o_ok['t']['F_I_rotB_conj_r'], th_ok, 1e-4)
    rest = sum(v for k, v in o['t'].items() if k != 'F_I_rotB_conj_r')
    print(f"  доля F_I_rotB в сумме правой стенки: {float(o['t']['F_I_rotB_conj_r']/sum(o['t'].values())):.6f};  остальные 11 членов: {float(o['nk'](rest)):.4g} Н/кВт")
print('ИТОГ:', 'OK' if allok else 'FAIL')

## 3. Двойной (тройной) счёт: какие члены лишние

Для однородной линейной среды с вещественными $\varepsilon,\mu$ и свободными токами $\mathbf j_e=\sigma_e\mathbf E+\mathbf j_{ext}$, $\mathbf j_m=\sigma_m\mathbf H$
(Минковский, $\rho_e=\rho_m=0$ внутри при TE, $k_y=0$):

$$\partial_x\langle T^{M}_{xx}\rangle=\langle f^M_x\rangle=\frac{1}{2c}\mathrm{Re}\big(j_{e,y}B_z^*\big)+\frac{1}{2c}\mathrm{Re}\big(j_{m,z}D_y^*\big),\qquad
\langle T^M_{xx}\rangle=\frac{1}{16\pi}\mathrm{Re}\big[2E_xD_x^*+2H_xB_x^*-\mathbf E\cdot\mathbf D^*-\mathbf H\cdot\mathbf B^*\big],$$

$\langle\partial_t\mathbf g\rangle=0$ при вещественной $\omega$. Значит $F_{jmD}+p_{Bt,jeB}=\int_a^\infty\langle f^M_x\rangle dx=-\langle T^M_{xx}\rangle(a^+)$, а полная сила на тело,
окружённое вакуумом, $F_{right}=-\langle T_{xx}^{vac}\rangle(a^-)$ — **не зависит** от формулировки (Минковский/Абрагам/Эйнштейн–Лауб).
Остальные члены `calc_thrust` уже содержатся в скачке $[T_{xx}]_a$ (Смайт) или являются другой записью тех же сил:
$\langle\dot{\mathbf P}\times\mathbf B\rangle=-\langle\mathbf P\times\dot{\mathbf B}\rangle=c\langle\mathbf P\times\mathrm{rot}\,\mathbf E\rangle$ ⇒ $F_{P\times\mathrm{rot}E}\equiv F_{dyn,P,B}$;
а при $\mathbf P'=\frac{\varepsilon'-1}{4\pi}\mathbf E$ имеем $\dot{\mathbf P}'=-i\omega\frac{\varepsilon-1}{4\pi}\mathbf E+\sigma_e\mathbf E$ — ток проводимости
посчитан ещё раз (в левой стенке $\varepsilon=1$: $F_{P\times\mathrm{rot}E,l}=F_{dyn,P,B,l}=p_{Bt,jeB,l}$ — тройной счёт). Аналогично $\mathbf M'=\frac{\mu'-1}{4\pi}\mathbf H$
содержит магнитный ток $i\sigma_m\mathbf H/\omega$.

In [ ]:
allok = True
for w in [1e6, 5e6, 1e7]:
    o = nb_terms(w); t = o['t']; u = o['u']
    print(f"omega={w:.0e}")
    allok &= ok('  F_jmD + p_Bt_jeB = -T_r(a+)', t['F_jmD_conj_right'] + t['p_Bt_jeB_conj_right'], -o['Tr_a'], 1e-12)
    allok &= ok('  F_jmD + p_Bt_jeB + jump = -T_v(a) (= maxwell_right_wall)', t['F_jmD_conj_right'] + t['p_Bt_jeB_conj_right'] + t['f_jump_maxwell_right'], -o['Tv_a'], 1e-12)
    allok &= ok('  F_P_rotE_r = F_dyn_P_B_r (двойной счёт, без бага сопряжения)', nb_terms(w, conj_bug=False)['t']['F_P_rotE_conj_r'], t['F_dyn_P_B_r'], 1e-12)
    allok &= ok('  F_P_rotE_l = p_Bt_jeB_l (тройной счёт)', u['F_P_rotE_conj_l'], u['p_Bt_jeB_conj_left'], 1e-12)
    allok &= ok('  F_dyn_P_B_l = p_Bt_jeB_l (тройной счёт)', u['F_dyn_P_B_l'], u['p_Bt_jeB_conj_left'], 1e-12)
    allok &= ok('  p_Bt_jeB_l(поля ячейки 110) + je_ext = T_l(-a)', o['u_cell110'] + u['p_Bt_je_ext_B_conj_left'], o['Tl_ma'], 1e-6)
    print(f"  удвоение je в E_y_l ячейки 278: p_Bt_jeB_l(278)/p_Bt_jeB_l(110) - 1 = {float(u['p_Bt_jeB_conj_left']/o['u_cell110'] - 1):.3e}")
print('ИТОГ:', 'OK' if allok else 'FAIL')

## 4. Поля в `scan_omega_re` — не решение задачи

`calc_thrust` берёт `X4, ans = svd_X4(M4_d_cur)`: минимальный сингулярный вектор **однородной** матрицы; `B_ext·je0` не используется
(`je0_ext` входит только в формулы полей левой стенки). При $\omega\le10^7\ll c\,k_z=9.4\cdot10^8$ вакуумный зазор запределен
($K_v\simeq ik_z$), строки $E_y$-сшивки масштабированы как $\omega/(ck_z)\sim10^{-3}$, поэтому `residual≈1e-3` выглядит малым, но
**непрерывность $E_y$ нарушена на величину порядка самой $E_y$**. Проверка: $\det M_4\ne0$, $\sigma_{min}\sim\omega/(ck_z)$, относительная
невязка строк 3–4.

In [ ]:
for w in [1e6, 1e7]:
    m = media(w); M, Bv = M4(m); X, res, s = svd_X4(M)
    r = (M @ X).astype(np.complex128)
    rel34 = [abs(r[i])/max(abs(complex(M[i, j]*X[j])) for j in range(4)) for i in (2, 3)]
    print(f"omega={w:.0e}: sing.values={np.array2string(s, precision=3)}, omega/(c k_z)={float(LD(w)/(P['c']*pi/P['Lz'])):.3e}, "
          f"|det M4|={abs(complex(np.linalg.det(M.astype(np.complex128)))):.3e}")
    print(f"   относит. невязка сшивки E_y (строки 3,4): {rel34[0]:.3f}, {rel34[1]:.3f}  ->", 'FAIL (не решение)' if max(rel34) > 1e-6 else 'OK')

## 5. Источник с профилем собственной моды: секулярный член и множитель 2

Ячейки 21/75/81–115: $j_y=j_0e^{-iK_l(x+a)}$ подставлен в алгебраические формулы $E_y(H_z,j)$, но $H_z$ оставлен однородным.
Точное уравнение в левой стенке ($k_y=0$): $H_z''+K_l^2H_z=-\frac{4\pi}{c}\partial_xj_y=\frac{4\pi iK_l}{c}j_0e^{-iK_l(x+a)}$ — резонансная правая часть, поэтому

$$H_z=\Big(B_{H,zl}-\frac{2\pi j_0}{c}(x+a)\Big)e^{-iK_l(x+a)},\qquad
E_y=\frac{c\mu'\omega K_l\big(B_{H,zl}-\frac{2\pi j_0}{c}(x+a)\big)+2\pi i\mu'\omega j_0}{c^2k_z^2-\varepsilon'\mu'\omega^2}\,e^{-iK_l(x+a)}.$$

В граничном условии `eq_Ey_l` должно стоять $2\pi i\mu'\omega j_0/D$, а не $4\pi i\mu'\omega j_0/D$ (ячейка 232, `B4_K[2]`): **все амплитуды
вынужденного решения ноутбука (ячейка 477) завышены ровно вдвое**, а поля ноутбука в левой стенке не удовлетворяют $\partial_xE_y=\frac{i\omega}{c}\mu'H_z$.

In [ ]:
allok = True
for w in [1e6, 1.7988142094590323e11]:
    m = media(w); M, Bv = M4(m)
    X_nb = gauss_solve(M.copy(), Bv*LD(1))         # как ячейка 477: M4 X = B_ext (je0=1)
    X_half = gauss_solve(M.copy(), Bv*LD(1)/2)
    S = solve_driven(w, LD(1)); a = P['a']
    allok &= ok(f'omega={w:.3e}: B_H_zr(B_ext/2) = H_z(a) точного решения', abs(X_half[3]), abs(S['U'](a, 'r')['Hz']), 1e-20)
    print(f"   отношение B_H_zr(ноутбук)/B_H_zr(точно) = {complex(X_nb[3]/S['U'](a, 'r')['Hz']):.15f}")
    # невязка ОДУ в левой стенке (центральные разности), точное решение
    U = S['U']; h = LD(1e-3)/abs(m['Kl']); xs = [-a - k*LD(0.3)/abs(m['Kl']) for k in range(1, 6)]
    c = m['c']; ww = m['w']; ml = m['ml']; Kl = m['Kl']
    res = max(abs((U(x + h, 'l')['Ey'] - U(x - h, 'l')['Ey'])/(2*h) - I*ww/c*ml*U(x, 'l')['Hz'])/abs(I*ww/c*ml*U(x, 'l')['Hz']) for x in xs)
    print(f"   невязка Фарадея (точное решение): {float(res):.2e}  ->", 'OK' if res < 1e-5 else 'FAIL'); allok &= res < 1e-5
    # то же для полей ноутбука с je0 != 0
    vac, left, right = fields_nb(m, X_nb, LD(1))
    resn = max(abs((left(x + h)['Ey'] - left(x - h)['Ey'])/(2*h) - I*ww/c*ml*left(x)['Hz'])/abs(I*ww/c*ml*left(x)['Hz']) for x in xs)
    print(f"   невязка Фарадея (поля ноутбука, je0=1): {float(resn):.2e}  ->", 'FAIL (не решение)' if resn > 1e-3 else 'OK')
print('ИТОГ:', 'OK' if allok else 'FAIL')

## 6. Правильный баланс: объём = тензор, $P_{src}=P_{diss}$, суммарная сила = 0

Вынужденное решение (секулярный источник, $j_0=10^6$), интегралы по Гауссу–Лежандру до $40/\mathrm{Im}K$:

* Минковский: $\int_a^\infty\langle f^M_x\rangle dx=-\langle T^M_{xx}\rangle(a^+)$, $\int_{-\infty}^{-a}\langle f^M_x\rangle dx=\langle T^M_{xx}\rangle(-a^-)$ (с силой на $j_{ext}$);
* Эйнштейн–Лауб: $\langle f^{EL}_x\rangle=\langle(\mathbf M\cdot\nabla)H_x\rangle+\frac{1}{2c}\mathrm{Re}[(j_e+\dot P)_yH_z^*]+\frac{1}{2c}\mathrm{Re}[(j_m+\dot M)_zE_y^*]$,
  $\langle T^{EL}_{xx}\rangle=\frac{1}{16\pi}\mathrm{Re}[2B_xH_x^*-|\mathbf E|^2-|\mathbf H|^2]$, $\int_a^\infty\langle f^{EL}_x\rangle=-\langle T^{EL}_{xx}\rangle(a^+)$;
* полная сила на стенку $F_{r}=-\langle T^{vac}_{xx}\rangle(a)$, $F_l=\langle T^{vac}_{xx}\rangle(-a)$; в вакуумном зазоре $\partial_x\langle T_{xx}\rangle=0$ ⇒ $F_l+F_r=0$;
* $P_{src}=-\frac12\mathrm{Re}\int j_{ext}^*E_y\,dx=P_{diss}=\frac12\int(\sigma_e|E|^2+\sigma_m|H|^2)dx$ (по полубесконечным стенкам).

In [ ]:
allok = True
rows = []
for w in [1e6*k for k in range(1, 11)] + [1.7988142094590323e11]:
    r = balance(w)
    allok &= ok(f'omega={w:.4e}: Минковский  объём = -T_r(a+)', r['Fr_vol_M'], -r['Tr'], 1e-9)
    allok &= ok('              Минковский  объём левой = T_l(-a-)', r['Fl_vol_M'], r['Tl'], 1e-8)
    allok &= ok('              Эйнштейн-Лауб объём = -T^EL_r(a+)', r['Fr_vol_EL'], -r['Tr_EL'], 1e-9)
    allok &= ok('              P_src = P_diss', r['Psrc'], r['Pdiss'], 1e-10)
    rel_thr = abs(r['thrust'])/abs(r['Fr_tot']); print(f"              |F_l+F_r|/|F_r| = {float(rel_thr):.2e}  ->", 'OK' if rel_thr < 1e-8 else 'FAIL'); allok &= rel_thr < 1e-8
    nk = lambda F: float(F/r['Psrc']*LD(1e5))
    rows.append((w, nk(r['Fr_vol_M']), nk(r['Fr_surf']), nk(r['Fr_tot']), nk(r['Fl_tot']), nk(r['thrust']), float(r['Psrc'])))
print('ИТОГ:', 'OK' if allok else 'FAIL')
print(f"\n{'omega':>11} | {'объём F_r':>11} | {'скачок [T]_a':>12} | {'F_r=-T_v(a)':>12} | {'F_l':>12} | {'F_l+F_r':>10} | {'P_src':>10}   (Н/кВт, норм. на P_src; P_src в эрг/(с·см²) при j0=1e6)")
for row in rows:
    print(f"{row[0]:11.4e} | {row[1]:11.4e} | {row[2]:12.4e} | {row[3]:12.4e} | {row[4]:12.4e} | {row[5]:10.1e} | {row[6]:10.4e}")

## 7. Нормировка «на кВт»

В ноутбуке $W$ — джоулевы потери только в слоях $h_l=1.875\cdot10^{-4}$ см и $h_r=6.25\cdot10^{-3}$ см, а силы интегрируются до $\infty$.
При $\omega=10^6$ $\delta_l=1/\mathrm{Im}K_l=0.016$ см, $\delta_r=0.094$ см $\gg h$ — учтено ~12 % потерь. Правильно: делить на $P_{src}$
(= полные потери). Кроме того, при вынужденном режиме мощность пропорциональна $|j_0|^2$, а у SVD-полей нормировка $\|X\|=1$ произвольна.

In [ ]:
for w in [1e6, 1e7, 1.7988142094590323e11]:
    o = nb_terms(w); m = o['m']
    vac, left, right = fields_nb(m, o['X'], LD(1e6)); R = right(P['a']); L = left(-P['a'])
    Wr_inf = (P['se_r']*abs(R['Ey'])**2 + P['sm_r']*(abs(R['Hx'])**2 + abs(R['Hz'])**2))/2/(2*m['Kr'].imag)
    Wl_inf = (P['se_l']*abs(L['Ey'])**2)/2/(2*m['Kl'].imag)
    print(f"omega={w:.3e}: delta_l={float(1/m['Kl'].imag):.3e} см, delta_r={float(1/m['Kr'].imag):.3e} см, W(слои h)/W(полная) = {float(o['W']/(Wr_inf + Wl_inf)):.4f}")

## 8. Итог

| что | ноутбук | правильно |
|---|---|---|
| поля | `svd_X4` однородной $M_4$ (не решение при вещественной $\omega$) | $M_4X=B_{ext}j_0$, источник $2\pi$ (не $4\pi$) + секулярный член |
| сила на правую стенку | 12 слагаемых, из них $F_{I\times\mathrm{rot}B}$ с потерянным $\overline{\mu'_r}$ даёт $+3466/\omega_6^2$ Н/кВт | $F_r=-\langle T^{vac}_{xx}\rangle(a)$ ($=F_{jmD}+p_{Bt,jeB}+[T]_a$) |
| полная сила | $\sum$ ≈ правая стенка | $F_l+F_r=0$ (до округления) |
| мощность | потери в слоях $h$ | $P_{src}=P_{diss}$ |